# NovelReader T4 — Web + dịch + audio
Bản tối ưu chạy từ **ZIP đã sửa**, không clone/pull GitHub để tránh ghi đè bản vá.

1. Chọn **Runtime → Change runtime type → T4 GPU**.
2. Chuẩn bị ZIP đi cùng notebook này. Chạy lần lượt các cell; ở bước 1 tải ZIP lên.
3. Đợi kiểm tra **CT2 CUDA + Piper CUDA thực tế** thành công rồi mở Web.

Không có cam kết số ký tự/giây. Tốc độ phụ thuộc truyện, giọng, beam, batch và runtime.
**Colab không bền vững:** tải output/sao lưu trước khi ngắt phiên. Dữ liệu đang chạy nằm ở `/content`, không ghi trực tiếp vào Drive.
**Tunnel là URL công khai không có đăng nhập:** không chia sẻ link, không dùng tài liệu nhạy cảm; tắt cell server khi xong.

## 1. Nạp ZIP đã sửa (chỉ cần làm một lần mỗi phiên Colab)

In [ ]:
import os, sys, subprocess, zipfile, stat
from pathlib import Path
from google.colab import files

REPO_DIR = Path('/content/NovelReader_T4_Web')
if not (REPO_DIR / 'scripts/setup_colab_runtime.py').is_file():
    print('Chọn file NovelReader_T4_Web_Optimized.zip đi cùng notebook.')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise RuntimeError('Hãy tải lên đúng 1 file ZIP dự án.')
    with zipfile.ZipFile(archives[0]) as z:
        marker = [i for i in z.namelist() if i.endswith('scripts/setup_colab_runtime.py')]
        if len(marker) != 1:
            raise RuntimeError('ZIP không đúng bản vá T4 Web.')
        prefix = marker[0][:-len('scripts/setup_colab_runtime.py')]
        for info in z.infolist():
            if not info.filename.startswith(prefix):
                continue
            rel = info.filename[len(prefix):]
            if not rel:
                continue
            target = (REPO_DIR / rel).resolve()
            if not target.is_relative_to(REPO_DIR.resolve()) or stat.S_ISLNK(info.external_attr >> 16):
                raise RuntimeError('ZIP chứa đường dẫn không an toàn.')
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(info) as src, target.open('wb') as dst:
                    import shutil
                    shutil.copyfileobj(src, dst)
    del uploaded
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('Thư mục:', REPO_DIR)
subprocess.run(['nvidia-smi'], check=True)

## 2. Cài thư viện (không thay PyTorch CUDA có sẵn của Colab)
Chỉ dùng CT2 nên không cần tải lại PyTorch. Bước này cài requirements trước, sau đó gỡ hai bản ORT và chỉ cài lại ORT GPU.
Nếu trước đó đã import ONNX Runtime/Piper/CT2 trong kernel, hãy **Restart session**, rồi chạy lại từ bước 1. Không đổi runtime sang CPU.

In [ ]:
if any(module in sys.modules for module in ('onnxruntime', 'piper', 'ctranslate2')):
    raise RuntimeError('Đã import GPU runtime. Restart session trước khi cài lại để tránh giữ thư viện cũ trong bộ nhớ.')
subprocess.run([sys.executable, 'scripts/setup_colab_runtime.py'], check=True)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'espeak-ng'], check=True)

## 3. Cấu hình T4
Web mặc định batch **96** (cho phép chỉnh đến 256), CT2 nội bộ **96**, beam **2**. Thử beam 1 trong Web nếu chấp nhận đánh đổi chất lượng.
Pipeline audio có công tắc trong Web để so sánh với chế độ tuần tự. Nếu VRAM thiếu, giảm batch 96 → 64 → 32.
Piper CUDA là bắt buộc ở notebook này, không chuyển CPU âm thầm. Để chủ động thử CPU TTS: đặt PIPER_DEVICE=cpu và PIPER_REQUIRE_CUDA=0, khởi động lại server; smoke check bước 4 khi đó sẽ không phù hợp vì nó kiểm tra CUDA bắt buộc.

In [ ]:
import site
os.environ.update({
    'HACHIMIMT_WEB_BATCH_SIZE': '96',
    'HACHIMIMT_BATCH_SIZE': '96',
    'HACHIMIMT_CT2_BATCH_TYPE': 'tokens',
    'HACHIMIMT_CT2_WINDOW_MULTIPLIER': '16',
    'HACHIMIMT_INTER_THREADS': '1',
    'HACHIMIMT_COMPUTE_TYPE': 'int8_float16',
    'HACHIMIMT_FORCE_CT2_CUDA': '1',
    'HACHIMIMT_GPU_INDICES': '0',
    'HACHIMIMT_AUTO_ALL_GPUS': '0',
    'HACHIMIMT_TOKENIZE_WORKERS': str(min(os.cpu_count() or 2, 4)),
    'PIPER_DEVICE': 'cuda', 'PIPER_REQUIRE_CUDA': '1',
    'ALLOWED_HOSTS': '*', 'ALLOW_TUNNEL': '1', 'COLAB_GPU': '1',
    'NOVEL_DATA': str(REPO_DIR / 'data'),
    'NOVEL_MODELS': str(REPO_DIR / 'models/translation'),
    'TTS_VOICE_DIR': str(REPO_DIR / 'models/voices'),
    'NOVEL_OUTPUT_TEXT': str(REPO_DIR / 'output_text'),
    'NOVEL_OUTPUT_AUDIO': str(REPO_DIR / 'output_audio'),
    'PYTHONUNBUFFERED': '1',
})
# Export NVIDIA wheel libraries before starting child processes.
libdirs = []
for base in site.getsitepackages():
    libdirs.extend(str(p) for p in (Path(base) / 'nvidia').glob('*/lib') if p.is_dir())
os.environ['LD_LIBRARY_PATH'] = ':'.join(libdirs + [os.environ.get('LD_LIBRARY_PATH', '')])
for name in ('data/logs', 'models/translation', 'models/voices', 'output_text', 'output_audio'):
    (REPO_DIR / name).mkdir(parents=True, exist_ok=True)
print('Đã đặt cấu hình. Chạy bước 4 rồi bước 5.')

## 4. Tải model cần dùng và kiểm tra GPU thật
Chỉ tải giọng mặc định, không tải toàn bộ kho giọng. Script dịch một câu và tạo WAV ngắn; nó dừng nếu CT2/Piper không dùng CUDA.
Nếu bước này lỗi, xem thông báo CUDA/cuDNN; đừng tiếp tục chạy Web rồi tưởng là đã dùng GPU.

In [ ]:
subprocess.run([sys.executable, 'scripts/check_gpu.py',
                '--model', 'HachimiMT-60-QT', '--voice', 'ngochuyennew'], check=True)

## 5. Khởi động Web và tunnel
Không mở nhiều server cùng lúc. **Stop cell này để dừng**: tiến trình nhận SIGTERM, có thời gian hoàn tất audio hiện tại; sau thời hạn sẽ buộc dừng.
Log server lưu trong `data/logs/colab-server.log`, tunnel dùng file log riêng để không bị nghẽn pipe chưa đọc.

In [ ]:
import time, re, urllib.request, signal
from IPython.display import display, Markdown

for name in ('server_process', 'tunnel_process'):
    old = globals().get(name)
    if old is not None and old.poll() is None:
        raise RuntimeError('Server/tunnel cũ còn chạy. Dừng cell cũ trước.')
# Fail early if port 8000 is already occupied.
import socket
with socket.socket() as sock:
    sock.bind(('127.0.0.1', 8000))
cloudflared = Path('/content/cloudflared')
if not cloudflared.is_file():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o755)
server_log = REPO_DIR / 'data/logs/colab-server.log'
tunnel_log = REPO_DIR / 'data/logs/colab-tunnel.log'
server_process = tunnel_process = None
try:
    with server_log.open('w') as sf, tunnel_log.open('w') as tf:
        server_process = subprocess.Popen(
            [sys.executable, '-m', 'uvicorn', 'main:app', '--host', '127.0.0.1',
             '--port', '8000', '--workers', '1', '--no-access-log', '--timeout-graceful-shutdown', '120'],
            cwd=REPO_DIR, stdout=sf, stderr=subprocess.STDOUT)
        for _ in range(120):
            if server_process.poll() is not None:
                raise RuntimeError(server_log.read_text()[-6000:])
            try:
                with urllib.request.urlopen('http://127.0.0.1:8000/api/state', timeout=2):
                    break
            except Exception:
                time.sleep(1)
        else:
            raise RuntimeError('Web chưa sẵn sàng. Xem colab-server.log.')
        tunnel_process = subprocess.Popen(
            [str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8000'],
            stdout=tf, stderr=subprocess.STDOUT)
        public_url = None
        for _ in range(90):
            if tunnel_process.poll() is not None:
                raise RuntimeError(tunnel_log.read_text()[-4000:])
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', tunnel_log.read_text())
            if match:
                public_url = match.group(0)
                break
            time.sleep(1)
        if not public_url:
            raise RuntimeError('Không lấy được URL tunnel. Xem colab-tunnel.log.')
        display(Markdown(f'### [Mở NovelReader]({public_url})\nLink công khai — không chia sẻ. Batch Web 96; mặc định không tự gộp.'))
        with server_log.open() as log:
            while server_process.poll() is None and tunnel_process.poll() is None:
                line = log.readline()
                if line:
                    print(line, end='')
                else:
                    time.sleep(.2)
except KeyboardInterrupt:
    print('Đang dừng. Không đóng runtime cho tới khi hoàn tất...')
finally:
    for proc in (tunnel_process, server_process):
        if proc is not None and proc.poll() is None:
            proc.terminate()
    for proc in (tunnel_process, server_process):
        if proc is not None and proc.poll() is None:
            try:
                proc.wait(timeout=135)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
                print('Đã buộc dừng quá thời hạn; audio chưa xong sẽ được làm lại khi tiếp tục.')
    print('Đã dừng Web và tunnel.')

## 6. Tải kết quả sau khi dừng Web
Gói gồm text/audio và database checkpoint. Không đưa token/cấu hình Google Drive vào bản sao lưu này. Sao lưu chỉ nhất quán khi server đã dừng.

In [ ]:
if globals().get('server_process') is not None and server_process.poll() is None:
    raise RuntimeError('Dừng cell Web trước khi đóng gói checkpoint.')
from datetime import datetime
backup = Path('/content') / ('NovelReader_outputs_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '.zip')
with zipfile.ZipFile(backup, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ('output_text', 'output_audio'):
        for p in (REPO_DIR / folder).rglob('*'):
            if p.is_file() and not any(part.startswith('.') for part in p.relative_to(REPO_DIR).parts):
                z.write(p, p.relative_to(REPO_DIR))
    # SQLite backup API also includes committed WAL data.
    import sqlite3, tempfile
    db = REPO_DIR / 'data/novel.sqlite3'
    if db.exists():
        with tempfile.TemporaryDirectory() as temp:
            snapshot = Path(temp) / 'novel.sqlite3'
            with sqlite3.connect(db) as src, sqlite3.connect(snapshot) as dst:
                src.backup(dst)
            z.write(snapshot, 'data/novel.sqlite3')
print('Bản sao lưu:', backup.name)
files.download(str(backup))

## Đo tốc độ (tùy chọn, dừng Web trước)
Dùng `translation/core/benchmark_sweep.py` với cùng file TXT để so batch/beam/compute type. Đây là benchmark core dịch, không đại diện thời gian Web + TTS.
Trong log Web có `TIMING translation` và `TIMING audio` theo chương. So sánh cùng vài chương, cùng giọng, cùng thiết lập để tránh kết luận sai.